# Coverage Model: Endpoint Test

This notebook calls the **deployed coverage model** over HTTPS and checks that
it's reachable, secured and giving sensible answers.

### What the model does

The model answers one question:

> *Given a library branch this far from a census tract, in this direction, how
> much of that tract can people reach within a set travel time?*

It returns an **overlap proportion** between 0 (none of the tract is in reach)
and 1 (all of it is). Scoring every tract this way for a hypothetical new branch
shows how much coverage that branch would add, without calling a mapping
service or redoing the geometry.

### Where the model comes from

1. **Training:** step 5 of the `cpl-coverage` pipeline (in the *pipelines*
   workbench) trains an **XGBoost** regression model on real branch-to-tract
   coverage. It registers the model in MLflow and saves a copy in XGBoost's
   native format to `models/coverage-model/model.json` in the project's S3
   bucket.
2. **Packaging:** that file is built into a container image together with the
   reference data it was trained against (see `model-image/Containerfile`).
   The image tag is set in the chart's `values.yaml` (`mcpServer.modelImage`).
3. **Serving:** **KServe**, OpenShift AI's model-serving layer, runs the image
   behind an HTTPS endpoint as an *InferenceService* named `cov-1`.

### Before you start

- The model must be **deployed in this project as `cov-1`**, with token
  authentication turned on. The Helm chart doesn't deploy the model itself.
  From the OpenShift AI dashboard (*Models → Deploy model*), pick an
  MLServer-based serving runtime that supports XGBoost, then choose where the
  model comes from. Either source works:

  | Model source | What to enter | When to use it |
  |---|---|---|
  | **Container image** | The image URI from `values.yaml` (`mcpServer.modelImage`) | You want a fixed, versioned model. Its tag only changes when someone builds and publishes a new image. |
  | **S3 model file** | The existing connection **CPL Data** (the project bucket) with the path `models/coverage-model/` | You want whatever the most recent pipeline run trained, with no image build. |

  The S3 option works because pipeline step 5 saves the model in its own folder
  in the bucket. KServe downloads everything under the path you give it, and
  MLServer finds `model.json` inside. Keep in mind that each pipeline run
  overwrites that file, so the endpoint serves the new model the next time its
  pod restarts.
- The workbench must be logged in to OpenShift (it is by default), because the
  notebook uses `oc` to find the endpoint and get a token.

### Key terms

| Term | Meaning |
|---|---|
| **KServe** | The Kubernetes component that turns a saved model into a scalable web service |
| **InferenceService** | The KServe resource describing one deployed model: which files, which runtime, how it's exposed |
| **Serving runtime / MLServer** | The server program that loads the model file and answers requests. MLServer is an open-source runtime that understands XGBoost models. |
| **Open Inference Protocol ("v2")** | A standard set of REST endpoints for model servers: `/v2/health/ready`, `/v2/models/<name>`, `/v2/models/<name>/infer` |
| **Bearer token** | A credential sent in the `Authorization` header. Here it's your OpenShift login token. |

### Contents

1. The model's inputs
2. Setup
3. Find the endpoint
4. Health and authentication checks
5. Make a prediction
6. How coverage decays with distance
7. Sanity checks

## 1. The model's inputs

Each prediction takes **eight numbers, in a fixed order**. The server receives
a bare array of numbers, not named fields, so the position of each value is
what tells the model what it means. Swap two and the model silently answers
a different question.

| # | Feature | Meaning |
|---|---|---|
| 0 | `distance_km` | Straight-line distance from the branch to the tract's center point, in kilometers |
| 1 | `bearing_sin` | Sine of the compass direction from branch to tract |
| 2 | `bearing_cos` | Cosine of the same direction |
| 3 | `tract_area_sqkm` | Tract area in square kilometers. Bigger tracts are harder to cover completely. |
| 4 | `travel_time` | Travel time in minutes |
| 5 | `mode_driving` | 1 if driving, else 0 |
| 6 | `mode_cycling` | 1 if cycling, else 0 |
| 7 | `mode_walking` | 1 if walking, else 0 |

**Why direction is two numbers:** compass bearings wrap around, so 359° and 1°
are nearly the same direction but numerically far apart. Encoding the angle as
its sine and cosine places directions on a circle, where neighbors stay close.
Direction matters because the lakefront, rivers and the street grid make reach
uneven.

**Why mode is three columns:** this is *one-hot encoding*. Exactly one of the
three is 1, which lets the model treat each travel mode as its own category
instead of pretending they sit on a numeric scale.

## 2. Setup

- **`requests`** sends the HTTP calls to the model server.
- **`urllib3.disable_warnings`** silences the warning printed for every request
  made with `verify=False`. Cluster routes in test and sandbox environments
  often use certificates that aren't publicly trusted, so certificate checking
  is turned off for these calls.
- **`MODEL_NAME`** is the name of the InferenceService to test.
- **`FEATURES`** records the input order from the table above.

In [ ]:
import json
import math
import subprocess

import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_NAME = "cov-1"

FEATURES = [
    "distance_km", "bearing_sin", "bearing_cos", "tract_area_sqkm",
    "travel_time", "mode_driving", "mode_cycling", "mode_walking",
]

## 3. Find the endpoint

Rather than pasting in a URL, the notebook asks OpenShift for it:

- `oc get inferenceservice cov-1 -o jsonpath='{.status.url}'` reads the public
  address KServe assigned to the model. Because it's looked up each time, the
  notebook works in any project or cluster where `cov-1` is deployed.
- `oc whoami -t` returns the OpenShift token of the user logged into this
  workbench. The endpoint accepts it as proof of identity.
- `HEADERS` attaches that token to every request as `Authorization: Bearer …`.

In [ ]:
def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

URL = sh(f"oc get inferenceservice {MODEL_NAME} -o jsonpath='{{.status.url}}'")
TOKEN = sh("oc whoami -t")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

print("endpoint:", URL)

## 4. Health and authentication checks

Three calls, each checking something different:

| Call | Expected | Why it matters |
|---|---|---|
| Model metadata **without** a token | `401 Unauthorized` | Proves the endpoint is protected. If this succeeds, anyone who finds the URL can use the model, and that needs fixing before sharing it. |
| `GET /v2/health/ready` with a token | `200` | The server is up and the model has loaded. |
| `GET /v2/models/cov-1` with a token | `200` plus JSON | Shows the model's name, version, platform, and the names, shapes and data types of its inputs and outputs. |

The metadata response is a useful reference: the input name and data type it
lists are exactly what the prediction request in the next section has to use.

In [ ]:
anon = requests.get(f"{URL}/v2/models/{MODEL_NAME}", verify=False, timeout=30)
print(f"no token        -> {anon.status_code} {'(rejected, good)' if anon.status_code == 401 else '(AUTH NOT ENFORCED)'}")

ready = requests.get(f"{URL}/v2/health/ready", headers=HEADERS, verify=False, timeout=30)
print(f"health/ready    -> {ready.status_code}")

meta = requests.get(f"{URL}/v2/models/{MODEL_NAME}", headers=HEADERS, verify=False, timeout=30)
print(f"model metadata  -> {meta.status_code}")
print(json.dumps(meta.json(), indent=2))

## 5. Make a prediction

A prediction is a `POST` to `/v2/models/cov-1/infer` with a JSON body in the
Open Inference Protocol format:

```json
{"inputs": [{
    "name": "predict",
    "shape": [rows, 8],
    "datatype": "FP32",
    "data": [row 1 values..., row 2 values..., ...]
}]}
```

- **`shape`** says how to read the numbers: `rows` predictions with 8 features
  each. Several rows can go in one request, which is much faster than one call
  per row.
- **`datatype` `FP32`** means 32-bit floating-point numbers.
- **`data`** is every row's values laid end to end in one flat list. The
  server uses `shape` to cut it back into rows.
- The answer comes back under `outputs[0].data`, one predicted overlap per row.

Two helpers keep the rest of the notebook short:

- **`predict(rows)`** flattens the rows, sends the request, raises an error if
  the server returns one, and returns the list of predictions.
- **`features(...)`** builds one 8-value row from readable arguments. Bearing,
  tract area, travel time and mode have defaults (45°, 1 km², 10 minutes,
  walking), so a call like `features(0.3)` means "a 1 km² tract 300 m
  northeast of a branch, for a 10-minute walk".

The first test asks about exactly that tract. It's well within walking range,
so expect a high overlap.

In [ ]:
def predict(rows):
    """rows: list of 8-value feature lists. Returns list of predicted overlaps."""
    flat = [v for row in rows for v in row]
    body = {"inputs": [{
        "name": "predict",
        "shape": [len(rows), len(FEATURES)],
        "datatype": "FP32",
        "data": flat,
    }]}
    r = requests.post(f"{URL}/v2/models/{MODEL_NAME}/infer",
                      headers=HEADERS, json=body, verify=False, timeout=60)
    r.raise_for_status()
    return r.json()["outputs"][0]["data"]


def features(distance_km, bearing_deg=45.0, area_sqkm=1.0, travel_time=10, mode="walking"):
    rad = math.radians(bearing_deg)
    return [
        distance_km, math.sin(rad), math.cos(rad), area_sqkm, travel_time,
        1 if mode == "driving" else 0,
        1 if mode == "cycling" else 0,
        1 if mode == "walking" else 0,
    ]


print("a tract 300m from a branch, 10 min walk:")
print("  predicted overlap =", predict([features(0.3)])[0])

## 6. How coverage decays with distance

A single number can look plausible by luck. A curve is a much stronger check.
This cell asks the model about the same tract at ten distances from 100 m to
8 km, for each of the three travel modes the pipeline scores, and arranges the
answers into a table with one column per mode.

**What a healthy model shows:**

- Overlap starts **high** close to the branch.
- It **falls off steeply**, then reaches **about zero** near the real reach of
  each mode: roughly **1.8 km for a 10-minute walk** and **3.2 km for a
  5-minute drive**. Cycling at 10 minutes reaches further than walking.
- Faster modes stay high for longer.

A curve that rises with distance, never reaches zero, or puts walking above
driving points to a problem with the training data or the inputs.

In [ ]:
import pandas as pd

distances = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0]
modes = [("walking", 10), ("cycling", 10), ("driving", 5)]

records = []
for mode, tt in modes:
    preds = predict([features(d, travel_time=tt, mode=mode) for d in distances])
    for d, p in zip(distances, preds):
        records.append({"mode": f"{mode}/{tt}min", "distance_km": d, "predicted_overlap": p})

curve = pd.DataFrame(records).pivot(index="distance_km", columns="mode", values="predicted_overlap")
curve.round(4)

Here is the same table drawn as a line chart (pandas plotting, built on
matplotlib). The thin gray line marks zero, which makes any dip below zero
easy to spot.

In [ ]:
ax = curve.plot(marker="o", figsize=(9, 5))
ax.set_xlabel("distance from branch (km)")
ax.set_ylabel("predicted overlap")
ax.set_title("Predicted tract coverage vs distance")
ax.axhline(0, color="#999", linewidth=0.8)
ax.grid(alpha=0.3)

## 7. Sanity checks

These turn the expectations above into explicit pass/fail checks, using a
10-minute walk at 0.2 km, 1.0 km and 3.0 km. They describe how coverage works
in the real world, so they should hold however the model was trained. A failure
here is a real finding about the model, not a flaky test.

| Check | Why it should hold |
|---|---|
| Closer tract covered more than a distant one | Reach shrinks with distance |
| Predictions never increase from 0.2 → 1.0 → 3.0 km | Same idea, across three points |
| Nothing exceeds 1.0 | You can't cover more than all of a tract |
| No negative predictions | You can't cover less than none of it |

**Known behavior:** the model is a plain regression model with nothing forcing
its output into the 0–1 range, so just beyond the distance where coverage stops
it can return small negative numbers, such as −0.002. The pipeline clips its own
predictions to 0–1, but anyone calling this endpoint directly gets the raw value
and should clip it themselves. If the last check fails by a tiny margin, this is
why.

In [ ]:
walk = predict([features(d, mode="walking", travel_time=10) for d in [0.2, 1.0, 3.0]])
near, mid, far = walk

checks = {
    "closer tract covered more than distant one": near > far,
    "monotonic decay across 0.2 / 1.0 / 3.0 km": near >= mid >= far,
    "nothing exceeds 1.0": all(p <= 1.0 for p in walk),
    "no negative predictions": all(p >= 0 for p in walk),
}

for label, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")

print()
print("predictions at 0.2 / 1.0 / 3.0 km:", [round(p, 6) for p in walk])